# Hands-on TF-IDF + SMS Spam Classification

**Goal:** build a binary text-classification model that predicts whether an SMS is `ham` (legitimate) or `spam`.

This is a teaching notebook: each section explains **what we are doing, why we are doing it, and why the important parameters are chosen**. Work through it from top to bottom and explain the reasoning aloud as you code.

## What the pipeline does

Raw SMS text → train/test split → TF-IDF numerical features → Logistic Regression → predictions → evaluation.

### Dataset
We use the local `SMSSpamCollection` file supplied for this workshop. It is a tab-separated text file with two fields per row:
- label: `ham` or `spam`
- message: the SMS text

**No dataset download is performed.** Keep `SMSSpamCollection` in the same folder as this notebook, or in a `data` subfolder.

## 0. Set up the project and explain the imports

We import each library close to where it is first used so that the notebook is easier to teach.

- `Path`: handles file paths in a way that works across operating systems.
- `pandas`: reads the tab-separated file and lets us inspect the data.
- Later, `scikit-learn` will split the data, create TF-IDF features, train a model and evaluate it.

**What to say:** “Before modelling, I first make sure the input file is present and understand the format. A model is only as reliable as the data pipeline feeding it.”

### Code walkthrough — locating your local dataset

- `from pathlib import Path`: imports a path utility; it avoids manually joining folder names with `/` or `\\`.
- `PROJECT_DIR = Path.cwd()`: gets the folder from which Jupyter is running. This is the notebook's working directory, which may differ from the notebook's saved location in some IDE setups.
- `DATA_CANDIDATES = [...]`: creates two possible locations to check: beside the notebook's working directory, and inside a `data` subfolder.
- `next((path for path in ... if path.is_file()), None)`: checks candidates in order and returns the first existing file. If neither exists, it returns `None`.
- `if DATA_FILE is None`: catches the missing-file case before trying to read it.
- `searched_paths = ...`: joins the candidate paths into a readable multi-line message.
- `raise FileNotFoundError(...)`: stops execution with a useful explanation rather than allowing a confusing error later.
- `print(...resolve())`: displays the full absolute path of the file actually selected.

**Say while presenting:** “I am not downloading data. I am checking the two local locations where I might have placed the extracted file, and I fail early with a clear message if it is missing.”

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
DATA_CANDIDATES = [
    PROJECT_DIR / "SMSSpamCollection",
    PROJECT_DIR / "data" / "SMSSpamCollection",
]
DATA_FILE = next((path for path in DATA_CANDIDATES if path.is_file()), None)

if DATA_FILE is None:
    searched_paths = "\n".join(str(path) for path in DATA_CANDIDATES)
    raise FileNotFoundError(
        "SMSSpamCollection was not found. Put it beside the notebook "
        "or inside a 'data' folder. Checked:\n" + searched_paths
    )

print("Dataset found:", DATA_FILE.resolve())

## 1. Load the dataset and inspect it

The file has **no header row**, so we supply column names ourselves. The separator is a tab (`\t`), not a comma.

Important parameters:
- `sep="\t"`: tells pandas how fields are separated.
- `header=None`: prevents pandas from treating the first SMS as column headings.
- `names=[...]`: gives meaningful names to the two columns.
- `encoding="utf-8"`: decodes text consistently.
- `on_bad_lines="skip"`: skips malformed rows rather than stopping the read. We will inspect the resulting row count.

We clean only obvious structural issues here: missing values, unexpected labels, empty messages and duplicate label/message rows. We do **not** aggressively clean message text yet, because punctuation, numbers and wording can carry useful spam signals.

**What to say:** “This is a supervised learning problem because each message already has a label. The model learns patterns from labelled examples.”

### Code walkthrough — reading and cleaning the SMS file

- `import pandas as pd`: imports pandas and gives it the conventional short name `pd`.
- `pd.read_csv(...)`: reads the tab-separated text file into a DataFrame.
- `DATA_FILE`: passes the local file path found in the setup cell.
- `sep="\\t"`: each row's two fields are separated by a tab character.
- `header=None`: says the file does not contain a header row; otherwise pandas might treat the first SMS as column names.
- `names=["label", "text"]`: assigns names to the two columns: class label and message body.
- `encoding="utf-8"`: decodes text as UTF-8.
- `on_bad_lines="skip"`: skips malformed rows. This is defensive; inspect the raw shape and make sure the loaded count is plausible.
- `df.shape`: returns `(number_of_rows, number_of_columns)`.
- `df.head(10)`: displays the first ten rows for a quick visual check.
- `dropna(subset=[...])`: removes rows missing either the label or message; unrelated missing columns are not relevant here.
- `.copy()`: creates a separate DataFrame for the cleaned data and avoids chained-assignment ambiguity.
- `astype(str)`: ensures the column values are strings before string operations.
- `str.strip()`: removes leading/trailing whitespace.
- `str.lower()`: normalises labels so `HAM`, `Ham` and `ham` are treated consistently.
- `isin(["ham", "spam"])`: keeps only the two expected target classes.
- `df["text"] != ""`: removes empty message bodies.
- `drop_duplicates(subset=["label", "text"])`: removes repeated rows only when both label and text match.
- `reset_index(drop=True)`: rebuilds row numbering from zero after rows have been removed.

**Say while presenting:** “I am cleaning structural problems, not rewriting the language. I want to preserve potentially useful signals in the SMS text.”

In [ ]:
import pandas as pd

df = pd.read_csv(
    DATA_FILE,
    sep="\t",
    header=None,
    names=["label", "text"],
    encoding="utf-8",
    on_bad_lines="skip",
)

print("Raw shape:", df.shape)
display(df.head(10))

df = df.dropna(subset=["label", "text"]).copy()
df["label"] = df["label"].astype(str).str.strip().str.lower()
df["text"] = df["text"].astype(str).str.strip()
df = df[df["label"].isin(["ham", "spam"])]
df = df[df["text"] != ""]
df = df.drop_duplicates(subset=["label", "text"]).reset_index(drop=True)

print("Cleaned shape:", df.shape)
display(df.head())

### 1.1 Check the target distribution

We count how many examples belong to each class and calculate their percentages.

Why? Spam datasets are often imbalanced: there may be many more legitimate messages than spam. If we report accuracy alone, a model can look good simply by predicting the majority class.

**What to say:** “I check the class distribution before training so I know whether accuracy will tell the full story.”

### Code walkthrough — the class-distribution snippet in detail

This is the section shown in your screenshot. We want to know whether ham and spam are evenly represented.

- `df["label"]`: selects the label column as a pandas Series.
- `.value_counts()`: counts how many times each distinct label occurs. By default it sorts from most frequent to least frequent.
- `class_counts = ...`: stores the raw counts, for example the number of ham messages and the number of spam messages.
- `.value_counts(normalize=True)`: returns proportions instead of raw counts. Each count is divided by the total number of rows, so the proportions add up to 1 (or 100% after multiplying by 100).
- `* 100`: converts proportions such as `0.13` into percentages such as `13.0`.
- `.round(2)`: rounds the percentages to two decimal places for readability; it does not change the underlying dataset.
- `.rename("message_count")`: gives the resulting Series a descriptive name so the displayed table has a meaningful column heading.
- `.rename("percentage")`: names the percentage Series.
- `.to_frame()`: converts a Series (one-dimensional) into a DataFrame (two-dimensional), which displays neatly as a one-column table.
- `display(...)`: renders the DataFrame as a formatted notebook table.

**Why do this before training?** If ham is, for example, 87% of the data, a model that always predicts ham could already get about 87% accuracy while detecting no spam at all. This is why we later compare against a baseline and report spam precision, recall and F1—not accuracy alone.

**Say while presenting:** “The first table gives counts; the second gives percentages. The percentages make imbalance easier to understand, especially when comparing datasets of different sizes.”

In [ ]:
class_counts = df["label"].value_counts()
class_percentages = (df["label"].value_counts(normalize=True) * 100).round(2)

display(class_counts.rename("message_count").to_frame())
display(class_percentages.rename("percentage").to_frame())

## 2. Understand TF-IDF manually before using a library

TF-IDF gives a word a high weight when it appears often in a particular document but is relatively uncommon across the whole collection.

- **TF (term frequency):** how prominent is the term inside this document?
- **IDF (inverse document frequency):** how informative is the term across all documents?
- **TF-IDF:** `TF × IDF`.

For a small, transparent example we use three toy messages. These are only for explaining the arithmetic; the classifier later uses the full SMS dataset.

We use smoothed IDF:

`IDF(t) = log((1 + N) / (1 + df(t))) + 1`

- `N` = number of documents.
- `df(t)` = number of documents containing the term.
- `+1` smoothing avoids division by zero and keeps the calculation stable.
- The final `+1` keeps IDF positive and matches the common scikit-learn convention.

In this manual example TF is raw count divided by document length. Scikit-learn later applies L2 normalization by default, so the final feature values will be scaled.

### Code walkthrough — building a tiny corpus for manual IDF

- `import math`: gives us `math.log()` for the logarithm in the IDF formula.
- `Counter`: counts token occurrences in one document; it is used in the next cell.
- `toy_documents`: three short example documents. They are deliberately small so we can calculate TF-IDF by hand.
- `doc.lower()`: normalises uppercase/lowercase so `Payment` and `payment` would be the same token.
- `.split()`: splits this simple toy text on whitespace. This is a teaching simplification, not a production tokenizer.
- `set(...)`: removes repeated terms when creating the vocabulary.
- `sorted(...)`: makes the vocabulary order predictable and easier to inspect.
- `N = len(tokenized_docs)`: number of documents in the corpus; IDF is based on documents, not total word occurrences.
- `for term in vocabulary`: calculates IDF separately for each vocabulary term.
- `term in set(doc)`: checks whether the term appears in a document. Converting to a set means repeated appearances in the same document still count once for document frequency.
- `sum(...)`: adds those True/False checks (True counts as 1), producing document frequency `df(t)`.
- `math.log((1 + N) / (1 + document_frequency)) + 1`: smoothed IDF. The `+1` inside numerator and denominator prevents division by zero; the final `+1` keeps weights positive.
- `rows.append({...})`: stores each term's document frequency and IDF in a row for display.
- `round(idf, 4)`: limits displayed decimal places; it is for readability.

**Important distinction:** term frequency counts occurrences within a document; document frequency counts how many different documents contain the term. A word repeated five times in one message still contributes only one to document frequency for that message.

In [ ]:
import math
from collections import Counter

toy_documents = [
    "payment delayed",
    "payment not processed",
    "forex fee charged",
]

tokenized_docs = [doc.lower().split() for doc in toy_documents]
vocabulary = sorted(set(token for doc in tokenized_docs for token in doc))
N = len(tokenized_docs)

rows = []
for term in vocabulary:
    document_frequency = sum(term in set(doc) for doc in tokenized_docs)
    idf = math.log((1 + N) / (1 + document_frequency)) + 1
    rows.append({
        "term": term,
        "document_frequency": document_frequency,
        "smoothed_idf": round(idf, 4),
    })

display(pd.DataFrame(rows))

### 2.1 Calculate TF-IDF values for one document

For each term in the vocabulary:
1. Count the term in the selected document.
2. Divide by the number of tokens in that document to get TF.
3. Multiply TF by the term's IDF.

**Why do this manually?** It makes the vectorizer less of a black box. The computer will perform the same broad idea across thousands of messages and terms.

### Code walkthrough — manual TF and TF-IDF for one message

- `tokenized_docs[0]`: selects the first toy document.
- `Counter(selected_doc)`: counts each word in that selected document.
- `len(selected_doc)`: total number of tokens in the document; this is the denominator for our simple relative-frequency TF.
- `manual_rows = []`: creates an empty list to collect one output row per vocabulary term.
- `for term in vocabulary`: calculates a weight for every term, including terms absent from this selected document.
- `selected_counts[term]`: gets the term count; `Counter` returns zero for a term that is absent.
- `tf = ... / doc_length`: computes TF as count divided by total tokens in this document.
- `df_term = sum(...)`: counts how many corpus documents contain the term, counting at most once per document.
- `idf = ...`: calculates the same smoothed IDF as the previous cell.
- `tf * idf`: combines local prominence (TF) with corpus rarity (IDF).
- `round(..., 4)`: rounds only the displayed values.
- `manual_rows.append({...})`: records term, raw count, TF, IDF and the unnormalised TF-IDF score.
- `pd.DataFrame(manual_rows)`: turns the list of dictionaries into a table.

**Why does a term sometimes get zero TF-IDF?** If it is not present in the selected document, its count and TF are zero, so TF × IDF is zero even if its IDF is high.

**Note:** this is a simplified manual calculation. Scikit-learn's default vectorizer also applies L2 normalisation, so its final values are scaled.

In [ ]:
selected_doc = tokenized_docs[0]
selected_counts = Counter(selected_doc)
doc_length = len(selected_doc)

manual_rows = []
for term in vocabulary:
    tf = selected_counts[term] / doc_length
    df_term = sum(term in set(doc) for doc in tokenized_docs)
    idf = math.log((1 + N) / (1 + df_term)) + 1
    manual_rows.append({
        "term": term,
        "count_in_document": selected_counts[term],
        "TF": round(tf, 4),
        "IDF": round(idf, 4),
        "TF_IDF_before_normalization": round(tf * idf, 4),
    })

display(pd.DataFrame(manual_rows))

## 3. Separate training data from test data

We split the labelled messages before fitting the vectorizer or model.

- `test_size=0.20`: hold out 20% for a final, unseen evaluation; train on 80%.
- `random_state=42`: makes the split reproducible, so the same rows go into train/test each run.
- `stratify=y`: preserves approximately the same ham/spam proportions in both sets.
- `X`: message text (input); `y`: class label (target).

**Critical concept — data leakage:** the vectorizer must learn its vocabulary and IDF weights from training text only. If we fit it on all messages before splitting, information from the test set leaks into training.

**What to say:** “The test set simulates messages the model has never seen. We keep it separate until evaluation.”

### Code walkthrough — creating a fair train/test split

- `train_test_split`: imports scikit-learn's splitting function.
- `X = df["text"]`: defines the input features as message text.
- `y = df["label"]`: defines the target we want to predict.
- `test_size=0.20`: reserves 20% for final testing and uses 80% for training.
- `random_state=42`: makes the random split reproducible. The number 42 is not special; using the same seed lets colleagues reproduce your result.
- `stratify=y`: preserves approximately the same ham/spam proportions in both subsets. This matters when one class is less common.
- `X_train`, `y_train`: messages and labels used to learn the model.
- `X_test`, `y_test`: held-out messages and labels used only for evaluation.
- `len(...)`: prints the number of examples in each subset.
- `.value_counts(normalize=True)`: calculates class proportions within each subset.
- `.round(3)`: rounds proportions to three decimal places.
- `.rename("proportion").to_frame()`: gives the Series a clear column name and turns it into a displayable table.

**Say while presenting:** “We split first. Later the vectorizer learns its vocabulary and IDF from training messages only. If we let test messages influence those values, our evaluation would be less independent.”

In [ ]:
from sklearn.model_selection import train_test_split

X = df["text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Training messages:", len(X_train))
print("Testing messages:", len(X_test))
print("\nTraining class proportions:")
display(y_train.value_counts(normalize=True).round(3).rename("proportion").to_frame())
print("Testing class proportions:")
display(y_test.value_counts(normalize=True).round(3).rename("proportion").to_frame())

## 4. Convert messages into TF-IDF features

Machine-learning algorithms need numeric inputs, not raw sentences. `TfidfVectorizer` builds a vocabulary and converts each message into a sparse numerical vector.

Parameters and why we use them:
- `lowercase=True`: treats `FREE` and `free` as the same token, reducing duplicate features.
- `ngram_range=(1, 2)`: uses unigrams (single words) and bigrams (two-word sequences). Bigrams can preserve short phrases such as “claim now” or “not interested”.
- `min_df=2`: ignores terms appearing in fewer than two training messages. This can reduce one-off noise, though rare but useful terms may also be removed.
- `max_df=0.95`: ignores terms appearing in more than 95% of training documents; extremely common terms often provide little separation.
- `sublinear_tf=True`: replaces raw term frequency with `1 + log(tf)` so repeating a word many times has diminishing impact.
- `stop_words=None`: we deliberately do not remove English stop words. Words such as “not” can reverse meaning, and stop-word lists can remove useful context.
- `norm="l2"` (default, stated explicitly): scales each document vector to unit length, helping reduce the effect of message length.

We call `fit_transform` on training text: `fit` learns vocabulary and IDF; `transform` creates vectors. On test text we call only `transform`, using the training vocabulary and IDF.

### Code walkthrough — fitting TF-IDF without leaking test information

- `TfidfVectorizer`: imports the text-to-numeric transformer.
- `lowercase=True`: normalises case before tokenisation.
- `ngram_range=(1, 2)`: includes one-word features and two-word sequences. A bigram can represent a phrase such as `claim now`.
- `min_df=2`: ignores a feature appearing in fewer than two training documents. This reduces very rare one-off terms; it may also discard rare useful terms, so it is a tunable choice.
- `max_df=0.95`: ignores a feature appearing in more than 95% of training documents. Extremely common terms often provide little separation.
- `sublinear_tf=True`: uses `1 + log(tf)` rather than raw term count, reducing the extra influence of a term repeated many times.
- `stop_words=None`: does not automatically remove common English words. Words like `not` can be important to meaning.
- `norm="l2"`: scales each message vector to unit Euclidean length; this reduces the influence of message length.
- `fit_transform(X_train)`: `fit` learns vocabulary and IDF from training messages; `transform` converts those messages to sparse numeric vectors.
- `transform(X_test)`: uses the already learned training vocabulary and IDF on test messages. Do not call `fit` or `fit_transform` on the test set.
- `.shape`: shows `(number of messages, number of features)`.
- `len(vectorizer.vocabulary_)`: number of learned text features.
- `type(...)`: shows the sparse matrix representation.

**Say while presenting:** “The rows are messages and columns are terms or n-grams. Most entries are zero, so scikit-learn stores the matrix sparsely to save memory.”

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    stop_words=None,
    norm="l2",
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))
print("Sparse matrix type:", type(X_train_tfidf))

### 4.1 Inspect a few features

The TF-IDF matrices are sparse: most words do not appear in most messages, so storing only non-zero values saves memory.

We convert **one row only** to a dense array for teaching. Avoid converting a large full matrix with `.toarray()` because it can consume a lot of memory.

### Code walkthrough — inspecting one sparse TF-IDF row

- `get_feature_names_out()`: returns the vocabulary feature names in the same column order as the TF-IDF matrix.
- `X_train_tfidf[0]`: selects the first training message's sparse row.
- `.toarray()`: converts this one row to a normal NumPy array for easy display. We only do one row; converting a huge sparse matrix can consume substantial memory.
- `.ravel()`: flattens the one-row array into a one-dimensional vector.
- `.nonzero()[0]`: gets indices of features whose weights are not zero.
- The list comprehension pairs each non-zero feature name with its TF-IDF weight.
- `sorted(..., key=..., reverse=True)`: sorts features from highest weight to lowest weight.
- `X_train.iloc[0]`: displays the original first training message; `.iloc` selects by row position.
- `y_train.iloc[0]`: displays its known label.
- `pd.DataFrame(..., columns=[...])`: makes a readable feature/weight table.
- `.head(15)`: shows only the top 15 entries.

**Interpretation:** a high TF-IDF weight means the term is prominent in this message and relatively informative across the training corpus. It does not, by itself, mean the term indicates spam.

In [ ]:
feature_names = vectorizer.get_feature_names_out()
sample_row = X_train_tfidf[0].toarray().ravel()
nonzero_indices = sample_row.nonzero()[0]

sample_features = sorted(
    [(feature_names[i], sample_row[i]) for i in nonzero_indices],
    key=lambda item: item[1],
    reverse=True,
)

print("Example training message:", X_train.iloc[0])
print("Actual label:", y_train.iloc[0])
display(pd.DataFrame(sample_features, columns=["feature", "tfidf_weight"]).head(15))

## 5. Establish a simple baseline

Before celebrating a sophisticated model, compare it with a trivial baseline.

`DummyClassifier(strategy="most_frequent")` always predicts the most common class in the training set. It does not learn language patterns.

Why? If the dataset is imbalanced, a model that predicts `ham` for every message may achieve deceptively high accuracy. Our real model should do better than this baseline, especially on spam recall and F1.

We use `stratify` in the split, and the baseline is fitted only on training labels.

### Code walkthrough — why build a dummy baseline?

- `DummyClassifier`: imports a model that follows a simple rule instead of learning word patterns.
- `accuracy_score` and `classification_report`: metrics used to evaluate its predictions.
- `strategy="most_frequent"`: always predicts whichever label is most common in the training labels.
- `baseline.fit(...)`: learns which label is most frequent. The feature matrix is supplied to match the usual estimator interface, but this strategy does not learn from text features.
- `baseline.predict(...)`: predicts the majority label for every test message.
- `accuracy_score(y_test, ...)`: compares predictions with the true test labels and returns the fraction correct.
- `round(..., 4)`: rounds the printed accuracy to four decimal places.
- `classification_report(...)`: prints precision, recall, F1 and support for each class.
- `zero_division=0`: if a class has no predicted examples, precision/recall calculations that would otherwise be undefined are reported as zero instead of raising a warning.

**What to compare:** if the trained model barely beats this baseline, or has poor spam recall, its accuracy alone is not evidence of a useful spam filter.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train_tfidf, y_train)
baseline_predictions = baseline.predict(X_test_tfidf)

print("Baseline accuracy:", round(accuracy_score(y_test, baseline_predictions), 4))
print(classification_report(y_test, baseline_predictions, zero_division=0))

## 6. Train Logistic Regression

Despite its name, Logistic Regression is a classification algorithm. It learns weights for the TF-IDF features and estimates the probability of each class.

Parameters:
- `max_iter=1000`: maximum optimisation iterations. Text feature spaces can require more iterations to converge; this gives the solver room to finish.
- `class_weight="balanced"`: automatically gives more weight to the less frequent class. This can help the model pay attention to spam rather than favouring the majority class.
- `random_state=42`: makes stochastic aspects reproducible where applicable. With the default solver it may not materially change this run, but it keeps the experiment configuration explicit.
- `solver="liblinear"`: a reliable solver for this small-to-medium binary classification example.

We fit on training TF-IDF features and training labels only.

### Code walkthrough — training Logistic Regression

- `LogisticRegression`: imports a linear classification algorithm that learns a coefficient for each TF-IDF feature.
- `max_iter=1000`: allows up to 1,000 optimisation iterations. If the solver has not converged, increasing iterations can help; it does not guarantee better generalisation.
- `class_weight="balanced"`: automatically weights classes inversely to their frequencies in the training data, so the less common class has more influence on the loss.
- `random_state=42`: sets the seed for solver components that use randomness; with `liblinear` it may have little or no effect, but makes the configuration explicit.
- `solver="liblinear"`: selects an optimisation algorithm that works well for this binary classification example and relatively small/medium datasets.
- `model.fit(X_train_tfidf, y_train)`: learns coefficients from training feature vectors and their labels.
- The test data is not passed to `.fit()`, preserving it for unbiased evaluation.

**Say while presenting:** “TF-IDF gives us the numeric representation; Logistic Regression learns which weighted terms are associated with ham or spam.”

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42,
    solver="liblinear",
)

model.fit(X_train_tfidf, y_train)
print("Model training complete.")

## 7. Evaluate on unseen test messages

We now predict the held-out test set and evaluate it.

Metrics:
- **Accuracy:** fraction of all predictions that are correct.
- **Precision for spam:** of messages predicted as spam, how many really are spam? Low precision means legitimate messages may be incorrectly flagged.
- **Recall for spam:** of all actual spam messages, how many did we catch? Low recall means spam is being missed.
- **F1-score:** harmonic mean of precision and recall; useful when both matter.
- **Confusion matrix:** shows counts of correct and incorrect predictions.

For a spam filter:
- False positive = legitimate ham marked as spam.
- False negative = spam marked as legitimate ham.

Do not report only accuracy. Discuss the cost of each error for the intended application.

### Code walkthrough — evaluating the classifier

- `accuracy_score`: calculates the overall fraction of correct predictions.
- `classification_report`: calculates precision, recall, F1 and support for each label.
- `confusion_matrix`: counts actual-versus-predicted label combinations.
- `ConfusionMatrixDisplay`: creates a visual confusion matrix.
- `matplotlib.pyplot`: plotting library used to render that visual.
- `model.predict(X_test_tfidf)`: predicts labels for held-out messages.
- `accuracy_score(y_test, y_pred)`: compares predicted labels with known test labels.
- The formatted print statement shows accuracy both as a decimal and a percentage.
- `labels=["ham", "spam"]`: fixes the class order so the report and matrix are consistent.
- `zero_division=0`: makes undefined metric cases display as zero.
- `confusion_matrix(..., labels=...)`: rows are actual labels and columns are predicted labels, in the specified order.
- `pd.DataFrame(...)`: labels the matrix axes clearly.
- `ConfusionMatrixDisplay(...).plot(...)`: draws the matrix; `values_format="d"` prints integer counts.
- `plt.title(...)`: adds a descriptive chart title; `plt.show()` displays it.

**Read the matrix carefully:** with order `[ham, spam]`, top-left is correctly predicted ham; top-right is ham incorrectly flagged as spam (false positive); bottom-left is spam missed as ham (false negative); bottom-right is correctly detected spam.

**Business interpretation:** false positives can hide legitimate messages; false negatives allow spam through. Which is worse depends on the application.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
import matplotlib.pyplot as plt

y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")
print("\nClassification report:")
print(classification_report(y_test, y_pred, labels=["ham", "spam"], zero_division=0))

cm = confusion_matrix(y_test, y_pred, labels=["ham", "spam"])
display(pd.DataFrame(
    cm,
    index=["Actual ham", "Actual spam"],
    columns=["Predicted ham", "Predicted spam"],
))

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["ham", "spam"],
).plot(cmap="Blues", values_format="d")
plt.title("SMS Spam Classifier — Confusion Matrix")
plt.show()

## 8. Try the model on new messages

The same vectorizer used during training must transform new messages. Do not create and fit a new vectorizer for predictions, because that would create a different feature space.

We also show predicted probabilities. They are model estimates, not guaranteed certainty.

### Code walkthrough — classifying new SMS messages

- `new_messages`: three example messages not used to train the model.
- `vectorizer.transform(new_messages)`: converts them into the same feature space learned from training data. We must reuse this fitted vectorizer.
- `model.predict(...)`: returns the predicted class label for each new message.
- `model.predict_proba(...)`: returns estimated class probabilities for each message.
- `model.classes_`: gives the column order used by `predict_proba`; we use it rather than assuming which probability column is ham or spam.
- `list(...).index("ham")`: finds the position of the ham class in that class order.
- `new_probabilities[:, ...]`: selects the probability column for all messages.
- `pd.DataFrame({...})`: combines message, predicted label and both class probabilities into a results table.
- `.round(3)`: rounds probability values for display.

**Caution:** probabilities are model estimates, not guaranteed certainty. A score of 0.90 should not automatically be described as a calibrated 90% chance unless calibration has been checked.

In [ ]:
new_messages = [
    "Congratulations! You have won a free prize. Call now to claim.",
    "Hey, are we still meeting for lunch today?",
    "URGENT! Your account has been selected for a cash reward.",
]

new_messages_tfidf = vectorizer.transform(new_messages)
new_predictions = model.predict(new_messages_tfidf)
new_probabilities = model.predict_proba(new_messages_tfidf)

prediction_results = pd.DataFrame({
    "message": new_messages,
    "predicted_label": new_predictions,
    "ham_probability": new_probabilities[:, list(model.classes_).index("ham")],
    "spam_probability": new_probabilities[:, list(model.classes_).index("spam")],
})
display(prediction_results.round(3))

## 9. Understand influential words

For binary Logistic Regression, the learned coefficient sign indicates which class a feature pushes towards. The class ordering is available in `model.classes_`.

A positive coefficient pushes the prediction towards `model.classes_[1]`; a negative coefficient pushes towards `model.classes_[0]`. These are model associations, not proof that a word causes spam.

We display the strongest positive and negative coefficients.

### Code walkthrough — interpreting Logistic Regression coefficients

- `numpy`: provides numerical array operations.
- `model.coef_[0]`: extracts the learned coefficient for every feature in this binary classifier.
- `np.argsort(coefs)`: returns feature indices sorted by coefficient value, from lowest to highest.
- `[-15:]`: takes the 15 largest coefficient indices.
- `[::-1]`: reverses them so the largest appears first.
- `[:15]`: takes the 15 smallest coefficient indices.
- `model.classes_`: shows the class order used internally by the model. In binary Logistic Regression, positive coefficients push towards `classes_[1]`; negative coefficients push towards `classes_[0]`.
- `feature_names[...]`: maps coefficient positions back to readable words or n-grams.
- `pd.DataFrame(...)`: displays each feature with its learned coefficient.

**Important:** coefficients show associations learned from this dataset and model. They do not prove that a word causes a message to be spam.

In [ ]:
import numpy as np

coefs = model.coef_[0]
top_positive_idx = np.argsort(coefs)[-15:][::-1]
top_negative_idx = np.argsort(coefs)[:15]

print("Classes:", model.classes_)
print("Features pushing towards class:", model.classes_[1])
display(pd.DataFrame({
    "feature": feature_names[top_positive_idx],
    "coefficient": coefs[top_positive_idx],
}))

print("Features pushing towards class:", model.classes_[0])
display(pd.DataFrame({
    "feature": feature_names[top_negative_idx],
    "coefficient": coefs[top_negative_idx],
}))

## 10. Save the trained model and vectorizer

At prediction time we need **both**:
- the fitted vectorizer, because it contains the vocabulary and IDF weights;
- the trained classifier, because it contains the learned decision weights.

We save them together in a dictionary using `joblib`. The output is written to the notebook's current working directory.

### Code walkthrough — saving what is needed for later predictions

- `joblib`: utility for saving/loading Python objects, commonly used with scikit-learn.
- `PROJECT_DIR / "...joblib"`: builds a path for the output file in the project working directory.
- `joblib.dump(...)`: serialises the Python object to disk.
- The dictionary contains both `vectorizer` and `model`. The vectorizer is essential because it stores the vocabulary and IDF weights; the classifier alone cannot convert raw text into the feature columns it expects.
- `MODEL_FILE`: tells `joblib.dump` where to save the bundle.
- `.resolve()`: prints the full output path.

**Production note:** only load joblib files from trusted sources; deserialising an untrusted file can execute code.

In [ ]:
import joblib

MODEL_FILE = PROJECT_DIR / "tfidf_sms_spam_pipeline.joblib"
joblib.dump(
    {"vectorizer": vectorizer, "model": model},
    MODEL_FILE,
)
print("Saved model bundle to:", MODEL_FILE.resolve())

## 11. Experiment: do bigrams help?

We compare unigrams-only `(1, 1)` against unigrams + bigrams `(1, 2)`. Bigrams can capture short phrases, but they also increase the number of features.

For a fair comparison:
- use the same train/test split;
- fit each vectorizer only on training text;
- train the same classifier configuration;
- compare spam precision, recall and F1, not just accuracy.

This is an experiment, not a guarantee that more n-grams will improve performance.

### Code walkthrough — comparing unigram and bigram settings

- `Pipeline`: chains text vectorisation and classification into one estimator, reducing the risk of accidentally applying different preprocessing at train and test time.
- `precision_recall_fscore_support`: computes class-specific precision, recall and F1.
- `experiment_rows = []`: collects one summary row per experiment.
- `[(1, 1), (1, 2)]`: tests unigrams only, then unigrams plus bigrams.
- `for ngram_setting ...`: repeats the experiment for each configuration.
- `Pipeline([...])`: first applies TF-IDF, then Logistic Regression.
- `ngram_range=ngram_setting`: passes the current n-gram choice into the vectorizer.
- The other TF-IDF and Logistic Regression parameters are held constant so the n-gram range is the main change.
- `fit(X_train, y_train)`: fits both steps using training text and labels; the pipeline internally fits TF-IDF only on training text.
- `predict(X_test)`: transforms test text with the fitted vectorizer and predicts labels.
- `labels=["spam"]`, `average=None`: asks for metrics specifically for spam and returns one value per requested label.
- `zero_division=0`: handles undefined metric cases safely.
- `experiment_rows.append(...)`: stores accuracy and spam-specific precision, recall and F1.
- `pd.DataFrame(...).round(4)`: displays the experiment comparison with four decimal places.

**How to interpret:** bigrams may help detect short phrases, but they increase feature count and may not improve every metric. Choose based on the evaluation and the cost of false positives versus false negatives.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_recall_fscore_support

experiment_rows = []

for ngram_setting in [(1, 1), (1, 2)]:
    experiment_pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            ngram_range=ngram_setting,
            min_df=2,
            max_df=0.95,
            sublinear_tf=True,
            stop_words=None,
            norm="l2",
        )),
        ("classifier", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
            solver="liblinear",
        )),
    ])

    experiment_pipeline.fit(X_train, y_train)
    experiment_pred = experiment_pipeline.predict(X_test)

    spam_precision, spam_recall, spam_f1, _ = precision_recall_fscore_support(
        y_test,
        experiment_pred,
        labels=["spam"],
        average=None,
        zero_division=0,
    )

    experiment_rows.append({
        "ngram_range": str(ngram_setting),
        "accuracy": accuracy_score(y_test, experiment_pred),
        "spam_precision": spam_precision[0],
        "spam_recall": spam_recall[0],
        "spam_f1": spam_f1[0],
    })

display(pd.DataFrame(experiment_rows).round(4))

## 12. Workshop wrap-up: explain the end-to-end reasoning

A concise explanation you can give the team:

1. “This is supervised binary text classification because each SMS has a ham/spam label.”
2. “I inspected the labels first to understand class imbalance.”
3. “I split the data before fitting TF-IDF to avoid test-data leakage.”
4. “TF-IDF converts text into numeric features, weighting terms by their importance within a message and across the corpus.”
5. “I used unigrams and bigrams so the model can use both individual words and short phrases.”
6. “I established a majority-class baseline before training Logistic Regression.”
7. “I evaluated on unseen messages and checked precision, recall, F1 and the confusion matrix because the two types of classification error have different costs.”
8. “Finally, I tested a feature configuration change and compared the results using the same split.”

### Questions to ask the team
- If spam recall improves but spam precision falls, is that acceptable for our use case?
- Would a false positive or a false negative be more costly?
- What happens if spam messages change over time (concept drift)?
- Would character n-grams help with misspellings or obfuscated spam terms?
- When might semantic embeddings outperform TF-IDF?

### Important limitations
- This is a classic bag-of-words-style representation; it does not fully understand meaning or context.
- The dataset is historical SMS data, so results may not transfer to modern messages or another organisation.
- A random train/test split measures performance on a held-out sample from the same dataset, not necessarily future real-world traffic.
- Probability outputs may need calibration before being used as confidence scores in production.